# Estimate missing bed capacity with XGBoost forests

Compare a single XGBoost random forest with a sequence of boosted forests. Both use the same audited capacity labels and saved facility partitions. The notebook marks missing predictors explicitly and removes records flagged as wrong by the audits.

In your Databricks Git folder, run `01_source_eda`, `02_facility_and_geography`, and `03_count_field_eda` from `non-ml/notebooks` in that order. Set `DATATHON_MODE` to `databricks` there to read the actual catalog tables. Use the same `DATATHON_OUTPUT_DIR` and snapshot date here. To work from local checkpoints, choose `export` mode. The stored checkpoints keep this notebook independent of the linking and audit steps.

In [ ]:
%pip install "xgboost==3.4.1" "scikit-learn==1.7.2" "wandb==0.31.2" "skops==0.16.0"

## Configure the inputs

The facility checkpoints are sufficient by default. Turn on density or the NFHS socioeconomic indicators when those inputs are available. The [density handoff](CLAUDE_HANDOFF.md) describes the district and PIN formats. Missing enrichment values remain eligible for modeling.

In [ ]:
import os
import sys
from pathlib import Path

INCLUDE_DENSITY = False
INCLUDE_NFHS = False
SAVE_PRIVATE_PREDICTIONS = False
LOG_ARTIFACT = True
SEED = 42
N_JOBS = 2

# For density, set exactly one of a Spark table or a CSV or Parquet path.
DENSITY_TABLE = ""
DENSITY_PATH = ""
DENSITY_KEY = "district"  # district or pin
DENSITY_COLUMNS = {
    "state_column": "state",
    "district_column": "district",
    "density_column": "population_density",
}
DENSITY_SOURCE = ""
DENSITY_VINTAGE = ""

relative = Path("ML/models with population density")
cwd = Path.cwd().resolve()
repo_override = os.environ.get("DATATHON_REPO_ROOT")
candidates = [Path(repo_override).expanduser()] if repo_override else [cwd, *cwd.parents]
root = next((path for path in candidates if (path / relative / "xgboost_capacity.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project Git folder or set DATATHON_REPO_ROOT.")
sys.path[:0] = [str(root / relative), str(root / "ML/notebooks"), str(root / "non-ml/src")]

import pandas as pd

from density_data import join_nfhs, join_population_density, prepare_district_keys
from facility_tool.workspace import Store, read_config
from wandb_models import log_xgboost_capacity
from xgboost_capacity import train_xgboost_capacity

config = read_config(root)
spark_session = globals().get("spark")
mode = os.environ.get("DATATHON_MODE", "databricks" if spark_session is not None else "export")
store = Store(root, mode, config["as_of"])
print(f"Reading the {mode} snapshot from {store.path}")

In [ ]:
facilities = store.read("linked_fdr")
capacity_audit = store.read("capacity_target_audit")
doctor_audit = store.read("number_doctors_target_audit")
location_records = store.read("location_records")
location_review = store.read("location_entity_review")

district_keys = None
if INCLUDE_NFHS or (INCLUDE_DENSITY and DENSITY_KEY == "district"):
    district_keys = prepare_district_keys(facilities, store.read("raw_pins"))

training = facilities
if INCLUDE_DENSITY:
    if bool(DENSITY_TABLE) == bool(DENSITY_PATH):
        raise ValueError("Set exactly one of DENSITY_TABLE and DENSITY_PATH.")
    if DENSITY_TABLE:
        if spark_session is None:
            raise ValueError("Use Databricks with DENSITY_TABLE or set DENSITY_PATH locally.")
        density = spark_session.table(DENSITY_TABLE).toPandas()
    elif Path(DENSITY_PATH).suffix.lower() == ".csv":
        density = pd.read_csv(DENSITY_PATH, dtype="string")
    elif Path(DENSITY_PATH).suffix.lower() == ".parquet":
        density = pd.read_parquet(DENSITY_PATH)
    else:
        raise ValueError("DENSITY_PATH must point to a CSV or Parquet file readable by Python.")
    training, density_report = join_population_density(
        training, density, keys=district_keys, key=DENSITY_KEY, **DENSITY_COLUMNS,
    )
    display(density_report)

if INCLUDE_NFHS:
    nfhs = store.read("raw_nfhs")
    if nfhs.empty and mode == "databricks" and spark_session is not None:
        nfhs = spark_session.table(config["tables"]["nfhs"]).toPandas()
    if nfhs.empty:
        raise ValueError("Supply the NFHS input to notebook 01 before enabling INCLUDE_NFHS.")
    training, nfhs_report = join_nfhs(training, nfhs, keys=district_keys)
    display(nfhs_report)

## Fit and check the two forests

Keep the existing facility split after applying the audit exclusions. Compare core facility features with the broader set of eligible predictors and choose the feature set and forest by calibration MAE. Numeric predictors retain their missing values and get a binary missing flag. Categorical predictors get a `missing` level. Compare held-out errors in beds against the training median. Scores describe facilities with observed capacity.

In [ ]:
result = train_xgboost_capacity(
    training, capacity_audit, doctor_audit, location_records, location_review,
    seed=SEED, n_jobs=N_JOBS,
)
print(f"Selected on calibration: {result['selected_model']} with {result['selected_feature_set']}")
print("Facilities by split:", result["facility_counts"])
print("Holdout fingerprint:", result["holdout_fingerprint"])
display(result["candidate_metrics"])
display(result["validation_metrics"])
display(result["test_metrics"])
display(result["feature_coverage"])
display(result["exclusions"])

## Inspect and optionally save estimates

Review estimates for facilities whose capacity was genuinely missing and passed the audits. These are estimated total bed counts. Keep any saved estimates in the private snapshot checkpoint directory. The source data and recommendation tables are independent of this notebook.

In [ ]:
estimates = result["predictions"]
print(f"Facilities with missing capacity estimated: {len(estimates):,}")
display(estimates.head(20))
if SAVE_PRIVATE_PREDICTIONS:
    destination = store.write_csv("xgboost_capacity_estimates", estimates)
    print(f"Saved estimates to {destination}")

## Save both forests with Weights & Biases

Each forest gets its own run and model artifact. The run includes validation and test scores, candidate feature comparisons, audit exclusions and training and calibration loss by round. The default mode is offline. See the [tracking setup](../README.md#experiment-tracking).

In [ ]:
if LOG_ARTIFACT:
    test = result["test_metrics"].set_index("model")
    evaluation = {
        "candidate_metrics": result["candidate_metrics"].to_dict(orient="records"),
        "validation_metrics": result["validation_metrics"].to_dict(orient="records"),
        "audit_exclusions": result["exclusions"].to_dict(orient="records"),
        "predictor_coverage": result["feature_coverage"].to_dict(orient="records"),
    }
    for name, model in result["models"].items():
        suffix = ("-density" if INCLUDE_DENSITY else "") + ("-nfhs" if INCLUDE_NFHS else "")
        run = log_xgboost_capacity(
            name="capacity-" + name.replace("_", "-") + suffix,
            model=model,
            preprocessor=result["preprocessor"],
            metrics=test.loc[name].to_dict(),
            baseline_metrics=test.loc["training_median"].to_dict(),
            history=result["training_history"][name],
            evaluation_tables=evaluation,
            config={
                "snapshot_as_of": config["as_of"],
                "input_mode": mode,
                "split_seed": SEED,
                "holdout_fingerprint": result["holdout_fingerprint"],
                "facility_counts": result["facility_counts"],
                "numeric_features": result["numeric_features"],
                "categorical_features": result["categorical_features"],
                "selected_on_calibration": name == result["selected_model"],
                "selected_feature_set": result["selected_feature_set"],
                "model_parameters": model.get_params(),
                "rounds_run": len(result["training_history"][name]),
                "include_density": INCLUDE_DENSITY,
                "include_nfhs": INCLUDE_NFHS,
                "density_source": (DENSITY_SOURCE or DENSITY_TABLE or "density file") if INCLUDE_DENSITY else None,
                "density_vintage": DENSITY_VINTAGE if INCLUDE_DENSITY else None,
                "density_key": DENSITY_KEY if INCLUDE_DENSITY else None,
            },
        )
        print(run)